# YAMNet 冻结嵌入提取（给队友用）

**用途**：在本 notebook 跑完后，`/kaggle/working/yamnet_embeddings/` 下会产出全部
`.npz` 嵌入文件 + `label_map.json`。队友在自己的 notebook 里 Add Input 挂载本 notebook
的 Output，读取这些文件即可开始训练。

**也可以**：把本 notebook 的 Cell 2-6 直接粘贴到队友 notebook 的最前面，一次 Run All，
YAMNet 算完嵌入直接写盘，队友的代码紧接着读盘训练——不用等两个 session。

```
OGG 文件 -> librosa 16k mono -> pad/clip 5s -> 峰值归一化
        -> [冻结 YAMNet] -> 帧嵌入 (T, 1024)
        -> mean+max+std 聚合 -> 3072 维向量
        -> 写 .npz 文件
```

**挂载要求**：
1. `processeddata-129834`（CSV 元数据：fold 划分、标签、类权重）
2. BirdCLEF 2021-2026 年度音频数据集（ogg 音频）
3. Accelerator: GPU（YAMNet 前向需要 GPU）

**产出文件**（`/kaggle/working/yamnet_embeddings/`）：

| 文件 | 内容 | 用途 |
|------|------|------|
| `embeddings.npz` | 全量 ~144K 条, 3072 维 | 训练/验证切片 |
| `embeddings_clean.npz` | 测试集 ~14K 条, 3072 维 | clean 基线 |
| `embeddings_5db.npz` | 测试集 ~14K 条, 3072 维 | 5dB 噪声评估 |
| `embeddings_0db.npz` | 测试集 ~14K 条, 3072 维 | 0dB 噪声评估 |
| `embeddings_m5db.npz` | 测试集 ~14K 条, 3072 维 | -5dB 噪声评估 |
| `label_map.json` | label2idx + idx2label + classes | 标签映射 |

**时间**：首次运行 ~20-60 min（嵌入提取）+ ~10-20 min（噪声嵌入）= ~0.5-1.5 小时。
（有缓存时二次运行 < 1 min）

**更新日志 (2026-07-19)**：
- 修复 YAMNet 输出 logits(521维) 被误当 frame_embeddings(1024维) 的致命 Bug
  （新增 `_select_frame_emb()` 按 dim==1024 健壮选取）
- 新增 YAMNet 批量调用探测（支持时 5-10x 前向加速）
- `EMB_BATCH_SIZE` 64 -> 128
- librosa `res_type="kaiser_fast"`（~2x 解码加速）
- `iterrows()` -> `to_dict('records')`（~10x 路径解析加速）
- 缓存维度校验：自动删除旧维度不匹配缓存


---
## Part 1/2: YAMNet 冻结嵌入提取

产出队友需要的全部 `.npz` 契约文件（基于 `yamnet_frozen_lightgbm_export.ipynb`
2026-07-19 最新版，已修复全部已知 Bug 并合并性能优化）。


In [ ]:
# 依赖: Kaggle 默认已装 tensorflow, tensorflow_hub, librosa, pandas, numpy
get_ipython().system("pip install resampy -q  # librosa kaiser_fast needs resampy")
import re
import os
import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import librosa

class Cfg:
    # YAMNet 输入: 16kHz 单声道 float32 [-1,1]
    SR = 16000
    CLIP_SECONDS = 5.0
    CLIP_SAMPLES = int(SR * CLIP_SECONDS)          # 80000

    # Kaggle 路径
    INPUT_BASE = "/kaggle/input"
    OUT_DIR = "/kaggle/temp/yamnet"
    EMB_DIR = "/kaggle/working/yamnet_embeddings"  # 给 LightGBM 队友 (契约目录)
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(EMB_DIR, exist_ok=True)

    # processeddata-129834 中的 CSV 文件名
    TRAIN_WEIGHTED_CSV = "02_train_full_weighted.csv"
    TEST_CSV = "03_test_holdout.csv"
    CLASS_WEIGHTS_CSV = "class_weights.csv"
    N_FOLDS = 5

    @staticmethod
    def fold_csvs(fold):
        return (f"cv_fold{fold}_train.csv", f"cv_fold{fold}_val.csv")

    # 音频目录候选 (不同年份命名不同)
    AUDIO_ROOT_CANDIDATES = ("train_audio", "train_short_audio")

    # YAMNet 模型 (首次联网下载, ~17MB)
    YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"

    # 嵌入聚合模式: "mean" (1024) / "mean_max" (2048) / "mean_max_std" (3072)
    # 契约推荐 mean_max_std -> 3072 维
    POOL_MODE = "mean_max_std"
    _POOL_FACTORS = {"mean": 1, "mean_max": 2, "mean_max_std": 3}
    EMBED_DIM = 1024 * _POOL_FACTORS[POOL_MODE]   # 3072

    # 嵌入提取 batch size (冻结前向, 已优化至 128)
    EMB_BATCH_SIZE = 128
    SEED = 42

    # 缓存路径 (内部用)
    EMB_CACHE = os.path.join(OUT_DIR, "embeddings_all.npz")
    NOISE_EMB_CACHE = os.path.join(OUT_DIR, "noise_embeddings.npz")
    LABEL_MAP_PATH = os.path.join(EMB_DIR, "label_map.json")


cfg = Cfg()


In [ ]:
def print_mounted_inputs(input_base=cfg.INPUT_BASE, max_per_dir=8):
    "列出 /kaggle/input 下挂载的数据集及其一层子目录。"
    path = Path(input_base)
    if not path.exists():
        print(f"[挂载] {path} 不存在")
        return
    entries = sorted(p for p in path.iterdir() if p.is_dir())
    if not entries:
        print(f"[挂载] {path} 下无子目录")
        return
    print(f"[挂载] {path} 下发现 {len(entries)} 个数据集:")
    for d in entries:
        sub = [c.name for c in d.iterdir() if c.is_dir()][:max_per_dir]
        print(f"  - {d.name}  子目录: {sub}")


def scan_kaggle_inputs(input_base=cfg.INPUT_BASE):
    "扫描 /kaggle/input 一次, 收集音频根目录与目标 CSV 路径。遇音频根剪枝。"
    target_csvs = {cfg.TRAIN_WEIGHTED_CSV, cfg.TEST_CSV, cfg.CLASS_WEIGHTS_CSV}
    for f in range(1, cfg.N_FOLDS + 1):
        target_csvs.update(cfg.fold_csvs(f))

    year2root = {}
    unknown_roots = []
    csv_paths = {}

    if not Path(input_base).exists():
        return year2root, unknown_roots, csv_paths

    for root, dirs, files in os.walk(input_base):
        base = os.path.basename(root)
        if base in cfg.AUDIO_ROOT_CANDIDATES:
            rp = Path(root)
            m = re.search(r"(20\d{2})", root)
            if m:
                year2root.setdefault(int(m.group(1)), rp)
            else:
                if rp not in unknown_roots:
                    unknown_roots.append(rp)
            dirs[:] = []
            continue
        for f in files:
            if f in target_csvs and f not in csv_paths:
                csv_paths[f] = Path(root) / f

    for r in unknown_roots:
        year2root.setdefault(-1, r)
    return year2root, unknown_roots, csv_paths


def parse_source_years(s):
    "解析 source_year 字段 (int 或 str), 返回降序年份列表。"
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return []
    years = re.findall(r"20\d{2}", str(s))
    return sorted({int(y) for y in years}, reverse=True)


def resolve_audio_path(row, year2root, all_roots):
    "依据 filename / primary_label / source_year 定位音频文件。"
    fname = str(row["filename"]).replace("\\", "/")
    primary_label = str(row["primary_label"])
    basename = os.path.basename(fname)

    roots_to_try = []
    for y in parse_source_years(row.get("source_year")):
        r = year2root.get(y)
        if r is not None and r not in roots_to_try:
            roots_to_try.append(r)
    for r in all_roots:
        if r not in roots_to_try:
            roots_to_try.append(r)

    for root in roots_to_try:
        for cand in (os.path.join(root, fname),
                     os.path.join(root, primary_label, basename),
                     os.path.join(root, basename)):
            if cand and os.path.exists(cand):
                return cand
    return None


def build_label_map(csv_paths):
    "构建 label2idx / idx2label (train + val_fold1 + test 并集)。"
    label_frames = []
    for fname in [cfg.TRAIN_WEIGHTED_CSV, cfg.TEST_CSV, cfg.fold_csvs(1)[1]]:
        if fname in csv_paths:
            label_frames.append(
                pd.read_csv(csv_paths[fname], usecols=["primary_label"])["primary_label"])
    classes = sorted(pd.concat(label_frames).astype(str).unique().tolist())
    label2idx = {c: i for i, c in enumerate(classes)}
    idx2label = {i: c for c, i in label2idx.items()}

    with open(cfg.LABEL_MAP_PATH, "w", encoding="utf-8") as f:
        json.dump({"label2idx": label2idx,
                   "idx2label": {str(k): v for k, v in idx2label.items()},
                   "classes": classes}, f, ensure_ascii=False, indent=2)
    print(f"[标签] {len(classes)} 类, 映射已存: {cfg.LABEL_MAP_PATH}")
    return classes, label2idx, idx2label

# ---- 执行扫描 + 标签映射 ----
print_mounted_inputs(cfg.INPUT_BASE)
year2root, unknown_roots, csv_paths = scan_kaggle_inputs(cfg.INPUT_BASE)
all_roots = list(year2root.values())
print(f"\n[音频根] {len(year2root)} 个年度根:")
for y, r in sorted(year2root.items(), key=lambda kv: str(kv[0])):
    print(f"  {y} -> {r}")
if not year2root:
    raise SystemExit("[错误] 未发现 train_audio/train_short_audio 目录。请确认已挂载 BirdCLEF 音频数据集。")
print(f"\n[CSV] 找到 {len(csv_paths)} 个 CSV:")
for f, p in sorted(csv_paths.items()):
    print(f"  {f} -> {p}")

classes, label2idx, idx2label = build_label_map(csv_paths)


In [ ]:
# ============================================================
# 流式波形加载: tf.py_function 包装 librosa
# ============================================================
def _load_wf_py(path):
    "Python 函数: 读 OGG -> 16k mono -> pad/clip 5s -> 峰值归一化。"
    p = path.numpy().decode("utf-8")
    y, _ = librosa.load(p, sr=cfg.SR, mono=True, res_type="kaiser_fast")
    target = cfg.CLIP_SAMPLES
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        start = (len(y) - target) // 2
        y = y[start:start + target]
    peak = np.max(np.abs(y)) + 1e-9
    return (y / peak).astype(np.float32)

def load_waveform_tf(path):
    "tf.data wrapper: path tensor -> waveform tensor [CLIP_SAMPLES]。"
    wf = tf.py_function(_load_wf_py, [path], tf.float32)
    wf.set_shape([cfg.CLIP_SAMPLES])
    return wf

# ============================================================
# 冻结 YAMNet 嵌入提取器 (mean / mean_max / mean_max_std)
# ============================================================
def _pool_frames(frames):
    "对 (B, T, 1024) 帧嵌入按 cfg.POOL_MODE 聚合。返回 (B, D)。"
    mean = tf.reduce_mean(frames, axis=1)                # (B, 1024)
    if cfg.POOL_MODE == "mean":
        return mean
    mx = tf.reduce_max(frames, axis=1)                    # (B, 1024)
    if cfg.POOL_MODE == "mean_max":
        return tf.concat([mean, mx], axis=1)             # (B, 2048)
    std = tf.math.reduce_std(frames, axis=1)              # (B, 1024)
    return tf.concat([mean, mx, std], axis=1)            # (B, 3072)

def build_embedding_extractor():
    "构建冻结 YAMNet KerasLayer + 帧聚合 -> D 维嵌入模型。"
    yamnet_layer = hub.KerasLayer(
        cfg.YAMNET_HANDLE,
        trainable=False,
        name="yamnet_frozen",
    )

    # ---- 从 YAMNet 输出中选出 1024 维 frame_embeddings ----
    def _select_frame_emb(out):
        "从 dict/tuple/single tensor 中选 last_dim==1024 的 frame_embeddings。"
        def _is_1024(v):
            shp = getattr(v, "shape", None)
            if shp is None or len(shp) < 1:
                return False
            return int(shp[-1]) == 1024
        if isinstance(out, dict):
            for k in ("frame_embeddings", "embedding", "embeddings"):
                if k in out and _is_1024(out[k]):
                    return out[k]
            for v in out.values():
                if _is_1024(v):
                    return v
            raise ValueError(
                "YAMNet dict 输出中找不到 1024 维 frame_embeddings: "
                "keys=" + str(list(out.keys())) + ", shapes="
                + str([(k, getattr(v, "shape", None)) for k, v in out.items()])
            )
        if isinstance(out, (list, tuple)):
            for v in out:
                if _is_1024(v):
                    return v
            return out[0]
        return out

    # ---- 探测: YAMNet KerasLayer 是否接受批量 (B, T) 输入? ----
    # 批量调用可让 GPU 一次前向处理整 batch, 比 per-sample 串行快 5-10x.
    BATCHED = False
    try:
        _probe_x = tf.zeros((2, cfg.CLIP_SAMPLES), dtype=tf.float32)
        _probe_out = yamnet_layer(_probe_x)
        _probe_fe = _select_frame_emb(_probe_out)
        if _probe_fe is not None and len(_probe_fe.shape) >= 2 and int(_probe_fe.shape[0]) == 2:
            BATCHED = True
    except Exception as e:
        print(f"  [probe] 批量调用探测失败: {type(e).__name__}: {e}")
        BATCHED = False
    if BATCHED:
        print("[YAMNet] 批量调用: 支持 (单次前向处理整 batch, 最优路径)")
    else:
        print("[YAMNet] 批量调用: 不支持, 回退到 tf.map_fn(parallel_iterations=8)")

    def _yamnet_batch(x):  # x: (B, 80000) -> (B, T, 1024)
        if BATCHED:
            # 最优: 一次批量前向
            return _select_frame_emb(yamnet_layer(x))   # (B, T, 1024)
        # 回退: tf.map_fn 并行 (比 tf.vectorized_map 更高效)
        def _per_sample(w):  # w: (80000,) -> (T, 1024)
            return _select_frame_emb(yamnet_layer(w))
        return tf.map_fn(_per_sample, x, parallel_iterations=8,
                         fn_output_signature=tf.TensorSpec(
                             shape=(None, 1024), dtype=tf.float32),
                         name="yamnet_per_sample")

    inp = tf.keras.Input(shape=(cfg.CLIP_SAMPLES,), dtype=tf.float32, name="waveform")
    emb = tf.keras.layers.Lambda(_yamnet_batch, name="yamnet_call")(inp)  # (B, T, 1024)
    if len(emb.shape) == 3:
        pooled = tf.keras.layers.Lambda(_pool_frames, name="frame_pool")(emb)
    else:
        pooled = emb
    model = tf.keras.Model(inp, pooled, name="yamnet_emb_extractor")
    print(f"[YAMNet] 冻结嵌入提取器: POOL_MODE={cfg.POOL_MODE}, "
          f"输出维度={model.output_shape} (D={cfg.EMBED_DIM})")
    return model, yamnet_layer

# ============================================================
# 流式批量嵌入计算
# ============================================================
def compute_embeddings_streaming(filepaths, emb_model, batch_size=cfg.EMB_BATCH_SIZE):
    "流式计算嵌入: paths -> tf.data -> batch -> YAMNet 前向 -> (N, D)。"
    filepaths = list(filepaths)
    n = len(filepaths)
    print(f"  [emb] 流式计算 {n} 条 ...")
    if n == 0:
        return np.zeros((0, cfg.EMBED_DIM), dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(load_waveform_tf, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    embs = []
    for i, batch in enumerate(ds):
        emb = emb_model(batch, training=False).numpy()
        embs.append(emb)
        done = min((i + 1) * batch_size, n)
        if (done % 500 < batch_size) or done == n:
            print(f"    [emb] {done}/{n} ({100*done/n:.1f}%)")
    return np.concatenate(embs, axis=0).astype(np.float32)

print("[YAMNet] 构建冻结嵌入提取器 ...")
emb_model, yamnet_layer = build_embedding_extractor()


In [ ]:
# ============================================================
# 一次性嵌入缓存 (全数据集唯一文件, 内部用)
# ============================================================
def build_embedding_cache(csv_paths, year2root, all_roots, emb_model):
    "一次性为全部 CSV 中出现的唯一文件计算嵌入, 缓存到 .npz。"
    cache_path = cfg.EMB_CACHE
    if Path(cache_path).exists():
        d = np.load(cache_path, allow_pickle=True)
        cached_dim = int(d["emb"].shape[1]) if d["emb"].ndim == 2 else -1
        if cached_dim != cfg.EMBED_DIM:
            print(f"[缓存] 维度不匹配 (cached={cached_dim} vs cfg.EMBED_DIM={cfg.EMBED_DIM}), "
                  f"删除旧缓存并重算: {cache_path}")
            os.remove(cache_path)
        else:
            print(f"[缓存] 读取: {cache_path}")
            fn2emb = {str(fn): d["emb"][i] for i, fn in enumerate(d["filenames"])}
            fn2label = {str(fn): str(lab) for i, (fn, lab)
                        in enumerate(zip(d["filenames"], d["labels_str"]))}
            print(f"  缓存命中 {len(fn2emb)} 条, dim={cached_dim}")
            return fn2emb, fn2label


    fn2path = {}
    fn2label = {}
    csv_files = [cfg.TRAIN_WEIGHTED_CSV, cfg.TEST_CSV]
    for f in range(1, cfg.N_FOLDS + 1):
        csv_files.extend(cfg.fold_csvs(f))

    for fname in csv_files:
        if fname not in csv_paths:
            continue
        df = pd.read_csv(csv_paths[fname])
        for row in df.to_dict('records'):
            fn = str(row["filename"])
            if fn in fn2path:
                continue
            p = resolve_audio_path(row, year2root, all_roots)
            if p is not None:
                fn2path[fn] = str(p)
                fn2label[fn] = str(row["primary_label"])

    fn_list = list(fn2path.keys())
    path_list = [fn2path[fn] for fn in fn_list]
    label_list = [fn2label[fn] for fn in fn_list]
    print(f"[缓存] {len(fn_list)} 个唯一文件, 开始流式计算嵌入 ...")
    print(f"  预计耗时 ~{len(fn_list)//400//60}h{len(fn_list)//400%60}m "
          f"(按 400 条/分钟估算)")

    X = compute_embeddings_streaming(path_list, emb_model)

    # 维度契约: 必须 == cfg.EMBED_DIM (1024/2048/3072).
    # 若为 521/1563, 说明 _per_sample 误取了 logits 而非 frame_embeddings.
    if X.ndim != 2 or X.shape[1] != cfg.EMBED_DIM:
        raise SystemExit(
            f"[契约失败] 嵌入维度 X.shape={X.shape} 与 cfg.EMBED_DIM={cfg.EMBED_DIM} 不符. "
            f"若 dim=521 或 1563, 说明 YAMNet 返回了 logits 而非 frame_embeddings; "
            f"请检查 _per_sample 的输出选择逻辑."
        )

    np.savez(cache_path,
             filenames=np.array(fn_list),
             labels_str=np.array(label_list),
             emb=X)
    print(f"[缓存] 已存: {cache_path} ({X.shape}, {X.nbytes/1e9:.2f} GB)")
    fn2emb = {fn: X[i] for i, fn in enumerate(fn_list)}
    return fn2emb, fn2label


# ============================================================
# 契约导出: yamnet_embeddings/embeddings.npz (filenames + emb)
# ============================================================
def export_contract_embeddings(fn2emb, fn2label):
    "按队友契约导出: yamnet_embeddings/embeddings.npz, key=filenames+emb。"
    fn_list = list(fn2emb.keys())
    emb_arr = np.stack([fn2emb[fn] for fn in fn_list]).astype(np.float32)
    out_path = os.path.join(cfg.EMB_DIR, "embeddings.npz")
    np.savez(out_path,
             filenames=np.array(fn_list),
             emb=emb_arr)
    print(f"[契约] {out_path}: {len(fn_list)} 条, emb shape={emb_arr.shape}")


# ============================================================
# 嵌入切片 (按 CSV 行顺序)
# ============================================================

SNR_TIERS = ["clean", "5dB", "0dB", "-5dB"]


def add_gaussian_noise(wf, snr_db, rng):
    "按指定 SNR 叠加高斯白噪声, 叠后重新归一化到 [-1,1]。"
    signal_power = np.mean(wf ** 2) + 1e-12
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), size=wf.shape).astype(np.float32)
    noisy = wf + noise
    peak = np.max(np.abs(noisy)) + 1e-9
    return (noisy / peak).astype(np.float32)


def _load_noisy_wf_py(path, snr, seed):
    "Python 函数: 读 OGG -> 16k mono -> pad/clip -> 归一化 -> 叠噪。"
    p = path.numpy().decode("utf-8")
    snr_val = float(snr.numpy())
    seed_val = int(seed.numpy())
    y, _ = librosa.load(p, sr=cfg.SR, mono=True, res_type="kaiser_fast")
    target = cfg.CLIP_SAMPLES
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        s = (len(y) - target) // 2
        y = y[s:s + target]
    peak = np.max(np.abs(y)) + 1e-9
    clean = (y / peak).astype(np.float32)
    rng = np.random.default_rng(seed_val)
    return add_gaussian_noise(clean, snr_val, rng)


def load_noisy_waveform_tf(path, snr, seed):
    "tf.data wrapper: (path, snr, seed) -> noisy waveform [CLIP_SAMPLES]。"
    wf = tf.py_function(_load_noisy_wf_py, [path, snr, seed], tf.float32)
    wf.set_shape([cfg.CLIP_SAMPLES])
    return wf


def build_noise_embeddings_streaming(df_test, year2root, all_roots,
                                     emb_model, fn2emb, label2idx):
    "流式计算测试集 3 档噪声嵌入; clean 直接复用主缓存。"
    cache_path = cfg.NOISE_EMB_CACHE
    if Path(cache_path).exists():
        print(f"[噪声嵌入] 读缓存: {cache_path}")
        d = np.load(cache_path, allow_pickle=True)
        return {k: d[k] for k in d.files}

    keep_rows = []
    keep_paths = []
    for _, row in df_test.iterrows():
        if str(row["filename"]) not in fn2emb:
            continue
        p = resolve_audio_path(row, year2root, all_roots)
        if p is not None:
            keep_rows.append(row)
            keep_paths.append(str(p))
    df_t = pd.DataFrame(keep_rows).reset_index(drop=True)
    n = len(keep_paths)
    print(f"[噪声嵌入] 测试集 {n} 条 (剔除 {len(df_test)-n} 条)")
    if n == 0:
        print("[噪声嵌入] 无可用样本, 跳过")
        return {}

    X_clean = np.stack([fn2emb[str(fn)] for fn in df_t["filename"]]).astype(np.float32)
    y_test = np.array([label2idx[str(l)] for l in df_t["primary_label"]], dtype=np.int64)
    test_filenames = df_t["filename"].values

    # 契约导出: clean 档 (测试集)
    np.savez(os.path.join(cfg.EMB_DIR, "embeddings_clean.npz"),
             filenames=test_filenames, emb=X_clean)
    print(f"[契约] embeddings_clean.npz: {n} 条, {X_clean.shape}")

    snr_tiers = [("5db", 5.0), ("0db", 0.0), ("m5db", -5.0)]
    noise_embs = {"X_clean": X_clean, "y_true": y_test,
                  "test_filenames": test_filenames}

    paths_arr = np.array(keep_paths)
    seeds_arr = np.arange(n, dtype=np.int32) + cfg.SEED

    for tier_name, snr_val in snr_tiers:
        print(f"  [噪声嵌入] tier {tier_name} (SNR={snr_val}) ...")
        snrs_arr = np.full(n, snr_val, dtype=np.float32)
        ds = tf.data.Dataset.from_tensor_slices((paths_arr, snrs_arr, seeds_arr))
        ds = ds.map(load_noisy_waveform_tf, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.batch(cfg.EMB_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        embs = []
        for i, batch in enumerate(ds):
            emb = emb_model(batch, training=False).numpy()
            embs.append(emb)
            done = min((i + 1) * cfg.EMB_BATCH_SIZE, n)
            if (done % 500 < cfg.EMB_BATCH_SIZE) or done == n:
                print(f"    {tier_name}: {done}/{n} ({100*done/n:.1f}%)")
        X_tier = np.concatenate(embs, axis=0).astype(np.float32)
        noise_embs[f"X_{tier_name}"] = X_tier

        # 契约导出: embeddings_5db.npz / embeddings_0db.npz / embeddings_m5db.npz
        np.savez(os.path.join(cfg.EMB_DIR, f"embeddings_{tier_name}.npz"),
                 filenames=test_filenames, emb=X_tier)
        print(f"[契约] embeddings_{tier_name}.npz: {n} 条, {X_tier.shape}")

    np.savez(cache_path, **noise_embs)
    print(f"[噪声嵌入] 内部缓存已存: {cache_path}")
    return noise_embs

print("[函数] 嵌入缓存 + 契约导出 + 噪声嵌入函数已定义")


In [ ]:
# ============================================================
# 执行 YAMNet 嵌入提取全流程（产出 .npz 契约文件）
# ============================================================
t0 = time.time()

# 阶段 1: 全量嵌入缓存 + 契约导出
print("\n" + "=" * 60)
print("  阶段 1/2: 冻结 YAMNet 全量嵌入提取")
print("=" * 60)
fn2emb, fn2label = build_embedding_cache(csv_paths, year2root, all_roots, emb_model)
print(f"[嵌入] 共 {len(fn2emb)} 条, 维度 {next(iter(fn2emb.values())).shape}")

export_contract_embeddings(fn2emb, fn2label)

# 阶段 2: 噪声嵌入 (测试集 4 档)
print("\n" + "=" * 60)
print("  阶段 2/2: 噪声嵌入提取 (测试集 clean/5dB/0dB/-5dB)")
print("=" * 60)
df_test = pd.read_csv(csv_paths[cfg.TEST_CSV])
noise_emb = build_noise_embeddings_streaming(
    df_test, year2root, all_roots, emb_model, fn2emb, label2idx)

elapsed = (time.time() - t0) / 60
print(f"\n{'='*60}")
print(f"[YAMNet 完成] 嵌入提取耗时 {elapsed:.1f} 分钟")
print(f"[契约输出] {cfg.EMB_DIR}")
print(f"  - embeddings.npz        全量 {len(fn2emb)} 条, {cfg.EMBED_DIM} 维")
if noise_emb:
    n_noise = len(noise_emb['test_filenames'])
    print(f"  - embeddings_clean.npz  测试集 {n_noise} 条")
    print(f"  - embeddings_5db.npz    测试集 {n_noise} 条")
    print(f"  - embeddings_0db.npz    测试集 {n_noise} 条")
    print(f"  - embeddings_m5db.npz   测试集 {n_noise} 条")
print(f"  - label_map.json        {len(classes)} 类标签映射")
print(f"\n下一步: 队友运行下一格开始 LightGBM 训练 (读 .npz -> 5 折训练)")


---
## Part 2/2: 队友如何加载嵌入

上面的 cell 跑完后，`.npz` 文件已在 `/kaggle/working/yamnet_embeddings/`。
下面的 cell **不执行任何操作**，只是展示加载代码。队友在自己的训练 cell 里加入
这几行即可，把原来的 MFCC 特征换成 YAMNet 嵌入。


In [ ]:
# ================= 加载示例（不执行, 供队友参考） =================
# 队友在自己的 notebook 里加入以下代码即可读取 YAMNet 嵌入:
#
# import numpy as np, json
#
# # ---- 1. 加载全量嵌入 (训练用) ----
# d = np.load("/kaggle/working/yamnet_embeddings/embeddings.npz", allow_pickle=True)
# fn2emb = {str(fn): d["emb"][i] for i, fn in enumerate(d["filenames"])}
# # fn2emb[fn] -> 3072 维 float32 向量, 如 fn2emb["abethr1/XC128013.ogg"].shape == (3072,)
#
# # ---- 2. 加载标签映射 ----
# with open("/kaggle/working/yamnet_embeddings/label_map.json") as f:
#     m = json.load(f)
#     label2idx = m["label2idx"]   # {"abethr1": 0, "abhori1": 1, ...}
#     classes = m["classes"]       # ["abethr1", "abhori1", ...]
#
# # ---- 3. 噪声嵌入 (测试集, 噪声鲁棒性评估用) ----
# d_clean = np.load("/kaggle/working/yamnet_embeddings/embeddings_clean.npz", allow_pickle=True)
# d_5db   = np.load("/kaggle/working/yamnet_embeddings/embeddings_5db.npz", allow_pickle=True)
# d_0db   = np.load("/kaggle/working/yamnet_embeddings/embeddings_0db.npz", allow_pickle=True)
# d_m5db  = np.load("/kaggle/working/yamnet_embeddings/embeddings_m5db.npz", allow_pickle=True)
# # 每个文件: d["filenames"] (N,) str + d["emb"] (N, 3072) float32
#
# # ---- 4. 在你的 split_xy 里, 用 fn2emb 替换原来的特征字典 ----
# # 原来可能长这样 (32 维 MFCC):
# #   Xs.append(fn2feat[fn])       # 32 维
# # 改成:
# #   Xs.append(fn2emb[fn])        # 3072 维
# # 其他代码 (lgb.train, fold 循环, accuracy 计算) 不需要改。
#
# # ---- 5. 按 filename 切片的完整示例 ----
# def split_xy(df, fn2emb, label2idx):
#     Xs, ys, fns = [], [], []
#     for _, r in df.iterrows():
#         fn = str(r["filename"])
#         if fn in fn2emb:
#             Xs.append(fn2emb[fn])
#             ys.append(label2idx[str(r["primary_label"])])
#             fns.append(fn)
#     return (np.array(Xs, dtype=np.float32),
#             np.array(ys, dtype=np.int64),
#             fns)

print("本 cell 仅展示加载代码, 不执行。请将上述代码复制到你的训练 cell 中。")
